In [1]:
import torch

In [2]:
torch.__version__

'2.7.0'

In [3]:
torch.cuda.is_available()

False

In [4]:
torch.backends.mps.is_available()

True

1. Understanding tensors
    1.1. Scalars, Vectors, Matrices and tensors
    1.2 Tensor data types
    1.3 Common Pytorch tensor operations
2. Seeing models as computational graphs
3. Computing gradients via autograd

In [5]:
t0d = torch.tensor(1)
print(t0d)

t1d = torch.tensor([1, 2, 3, 4])
print(t1d)

t2d = torch.tensor([[1, 2], 
                    [3, 4]])
print(t2d)

t3d = torch.tensor([[[1, 2], [3, 4]],
                     [[5, 6], [7, 8]]])
print(t3d)

tensor(1)
tensor([1, 2, 3, 4])
tensor([[1, 2],
        [3, 4]])
tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])


In [6]:
print(t0d.dtype)

torch.int64


In [7]:
t1df = torch.tensor([1.0, 2.0, 3.0])
print(t1df)
print(t1df.dtype) 

tensor([1., 2., 3.])
torch.float32


In [8]:
t2df = t2d.to(torch.float32)
print(t2df)
print(t2df.dtype)

tensor([[1., 2.],
        [3., 4.]])
torch.float32


In [9]:
t2 = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(t2)
print(t2.shape)

tensor([[1, 2, 3],
        [4, 5, 6]])
torch.Size([2, 3])


In [10]:
t3 = torch.tensor([[[1, 2, 3],[4, 5, 6]], 
             [[3, 4,5], [5, 6,7 ]]])
print(t3)
print(t3.dtype)
print(t3.shape)

tensor([[[1, 2, 3],
         [4, 5, 6]],

        [[3, 4, 5],
         [5, 6, 7]]])
torch.int64
torch.Size([2, 2, 3])


In [11]:
print(t2)
print(t2.reshape(3, 2))
print(t2.T)

tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2],
        [3, 4],
        [5, 6]])
tensor([[1, 4],
        [2, 5],
        [3, 6]])


In [12]:
print(t2.view(3, 2))
print(t2.view(1, 6))
print(t2.view(6))

tensor([[1, 2],
        [3, 4],
        [5, 6]])
tensor([[1, 2, 3, 4, 5, 6]])
tensor([1, 2, 3, 4, 5, 6])


In [13]:
print(t2.matmul(t2.T))

tensor([[14, 32],
        [32, 77]])


In [14]:
print(t2@t2.T)

tensor([[14, 32],
        [32, 77]])


In [15]:
import torch.nn.functional as F

In [16]:
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2])
b = torch.tensor([0.0]) 
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

In [17]:
a

tensor([0.9183])

In [18]:
y

tensor([1.])

In [19]:
loss

tensor(0.0852)

In [20]:
from torch.autograd import grad

In [21]:
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [22]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


## A mutlilayer perceptron with two hidden layers:

In [23]:
class NeuralNetwork(torch.nn.Module):
    
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30), 
            torch.nn.ReLU(), 

            # 2nd hidden layer:
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(), 

            #output layer:
            torch.nn.Linear(20, num_outputs), 

        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [24]:
model = NeuralNetwork(50, 3)

In [25]:
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [26]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of parameters", num_params)

Total number of parameters 2213


In [27]:
print(model.layers[0])

Linear(in_features=50, out_features=30, bias=True)


In [28]:
print(model.layers[0].weight)

Parameter containing:
tensor([[ 0.0162, -0.0854, -0.0794,  ...,  0.1291, -0.0031, -0.1049],
        [-0.0421, -0.0230,  0.0567,  ...,  0.1134,  0.0921, -0.0135],
        [-0.0468,  0.1344,  0.0942,  ..., -0.0298,  0.1229,  0.0651],
        ...,
        [ 0.1150, -0.0997, -0.0744,  ...,  0.1270, -0.0164,  0.0447],
        [ 0.0104,  0.0568, -0.0649,  ...,  0.0094,  0.1045, -0.1028],
        [ 0.0563,  0.0086,  0.0597,  ...,  0.0140,  0.1288, -0.0338]],
       requires_grad=True)


In [29]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [30]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [31]:
torch.manual_seed(123)
X = torch.rand((1, 50))
print(X)
out = model(X)
print("_______"*50)
print(out)


tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665, 0.1366, 0.1025, 0.1841,
         0.7264, 0.3153, 0.6871, 0.0756, 0.1966, 0.3164, 0.4017, 0.1186, 0.8274,
         0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826, 0.2745, 0.6584, 0.2775,
         0.8573, 0.8993, 0.0390, 0.9268, 0.7388, 0.7179, 0.7058, 0.9156, 0.4340,
         0.0772, 0.3565, 0.1479, 0.5331, 0.4066, 0.2318, 0.4545, 0.9737, 0.4606,
         0.5159, 0.4220, 0.5786, 0.9455, 0.8057]])
______________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________
tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [32]:
with torch.no_grad():
    out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [33]:
with torch.no_grad():
    out = torch.softmax(model(X), dim = 1)
print(out)

tensor([[0.3113, 0.3934, 0.2952]])


## Creating a small toy dataset:

In [34]:
X_train = torch.tensor([
    [-1.2,3.1], 
    [-0.9, 2.9], 
    [-0.5, 2.6], 
    [2.3, -1.1], 
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8], 
    [2.6, -1.6]
])
y_test = torch.tensor([0, 1])

In [35]:
from torch.utils.data import Dataset

In [36]:

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)    

In [37]:
print(len(train_ds))

5


In [38]:
from torch.utils.data import DataLoader

In [39]:
torch.manual_seed(123)

train_loader = DataLoader(
    dataset = train_ds, 
    batch_size = 2, 
    shuffle = True, 
    num_workers = 0
)

test_loader = DataLoader(
    dataset = test_ds, 
    batch_size = 2, 
    shuffle = True, 
    num_workers = 0 
)

In [40]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)


Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


In [41]:
train_loader = DataLoader(
    dataset = train_ds, 
    batch_size = 2, 
    shuffle = True, 
    num_workers = 0, 
    drop_last = True
)

In [42]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


### A typical Training Loop:

In [43]:
import torch.nn.functional as F

torch.manual_seed(123)

model = NeuralNetwork(num_inputs = 2, num_outputs = 2)

optimizer = torch.optim.SGD(model.parameters(), lr = 0.5)

num_epochs = 3
for epoch in range(num_epochs):
    model.train()

    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### Logging:

        print(f"Epoch:{epoch+1:03d}/{num_epochs:03d}"
              f"| Batch {batch_idx:03d}/{len(train_loader):03d}"
              f"| Train Loss: {loss:.2f}")
    model.eval()



Epoch:001/003| Batch 000/002| Train Loss: 0.75
Epoch:001/003| Batch 001/002| Train Loss: 0.65
Epoch:002/003| Batch 000/002| Train Loss: 0.44
Epoch:002/003| Batch 001/002| Train Loss: 0.13
Epoch:003/003| Batch 000/002| Train Loss: 0.03
Epoch:003/003| Batch 001/002| Train Loss: 0.00


In [44]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [45]:
outputs

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])

In [46]:
torch.set_printoptions(sci_mode = False)
probas = torch.softmax(outputs,dim = 1)
print(probas)

tensor([[    0.9991,     0.0009],
        [    0.9982,     0.0018],
        [    0.9949,     0.0051],
        [    0.0491,     0.9509],
        [    0.0307,     0.9693]])


In [47]:
predictions = torch.argmax(probas, dim = 1)
print(predictions)

tensor([0, 0, 0, 1, 1])
